# 🍃 Tea Leaf Disease Detection
### CNN with MobileNetV2 Transfer Learning


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os, json, random
from pathlib import Path
from PIL import Image

print('TF version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices("GPU")) > 0)

## 1. Dataset Exploration

In [ ]:
CLASSES = [
    'healthy', 'anthracnose', 'algal_leaf', 'bird_eye_spot',
    'brown_blight', 'gray_light', 'red_leaf_spot', 'white_spot'
]
DATA_DIR = '../data/train'

# Count images per class
counts = {}
for cls in CLASSES:
    path = os.path.join(DATA_DIR, cls)
    if os.path.exists(path):
        counts[cls] = len(os.listdir(path))

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(counts.keys(), counts.values(), color=plt.cm.Set2.colors, edgecolor='white')
ax.set_title('Image Count per Disease Class', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
plt.xticks(rotation=30, ha='right')
for bar, v in zip(bars, counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.show()
print(f'Total images: {sum(counts.values())}')

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, cls in zip(axes.flatten(), CLASSES):
    path = os.path.join(DATA_DIR, cls)
    if os.path.exists(path):
        imgs = os.listdir(path)
        img_path = os.path.join(path, random.choice(imgs))
        img = Image.open(img_path).resize((224, 224))
        ax.imshow(img)
        ax.set_title(cls.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.axis('off')
plt.suptitle('Sample Images — One Per Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 2. Data Augmentation Preview

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img

aug = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3]
)

# Pick one image
sample_dir = os.path.join(DATA_DIR, CLASSES[0])
sample_path = os.path.join(sample_dir, os.listdir(sample_dir)[0])
img = load_img(sample_path, target_size=(224, 224))
arr = np.expand_dims(img_to_array(img), 0)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes[0][0].imshow(img); axes[0][0].set_title('Original'); axes[0][0].axis('off')
gen = aug.flow(arr, batch_size=1)
for ax in list(axes.flatten())[1:]:
    aug_img = next(gen)[0].astype('uint8')
    ax.imshow(aug_img); ax.axis('off')
plt.suptitle('Data Augmentation Examples', fontsize=13)
plt.tight_layout(); plt.show()

## 3. Model Architecture

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, Input

base = MobileNetV2(input_shape=(224,224,3), include_top=False, weights='imagenet')
base.trainable = False

inp = Input(shape=(224,224,3))
x = base(inp, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(len(CLASSES), activation='softmax')(x)

model = models.Model(inp, out)
model.summary()

trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
total     = sum(np.prod(v.shape) for v in model.variables)
print(f'\nTrainable params : {trainable:,}')
print(f'Frozen params    : {total-trainable:,}')

## 4. Training History (Post-Training)

In [ ]:
# Run this after training to visualize saved history
import glob
for img_path in glob.glob('../results/history_*.png'):
    img = plt.imread(img_path)
    plt.figure(figsize=(14, 5))
    plt.imshow(img); plt.axis('off')
    plt.title(img_path.split('/')[-1], fontsize=12)
    plt.show()

## 5. Evaluation Results (Post-Training)

In [ ]:
# Load and display metrics
if os.path.exists('../results/classification_report.json'):
    with open('../results/classification_report.json') as f:
        report = json.load(f)
    print(f'Overall Accuracy  : {report["accuracy"]*100:.2f}%')
    print(f'Macro F1          : {report["macro avg"]["f1-score"]*100:.2f}%')
    print(f'Weighted F1       : {report["weighted avg"]["f1-score"]*100:.2f}%')
    
    # Per-class table
    import pandas as pd
    rows = []
    for cls in CLASSES:
        if cls in report:
            r = report[cls]
            rows.append({'Class': cls, 'Precision': f"{r['precision']:.3f}",
                         'Recall': f"{r['recall']:.3f}", 'F1': f"{r['f1-score']:.3f}",
                         'Support': int(r['support'])})
    print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Show confusion matrix & ROC
for p in ['../results/confusion_matrix.png', '../results/roc_curves.png', '../results/per_class_metrics.png']:
    if os.path.exists(p):
        plt.figure(figsize=(12, 8))
        plt.imshow(plt.imread(p)); plt.axis('off')
        plt.show()